In [1]:
import csv
import os
import sys
import pandas as pd
import numpy as np
from emmaemb.core import Emma
sys.path.append('/home/unix/vkrhk/EmmaEmb/EmmaEmb/analysis')
from constants import DATA_PATH, EMBEDDINGS_PATH, IMG_OUTPUT_PATH, EMB_SPACES, CRYPTOBENCH_TRAIN_DATASET, SCPDB_DATASET

## Undersample 
take size of cryptic binding sites (the smallest feature class) and undersample the other two classes (i.e., binding + non-binding):
1. binding: take N proteins from sc_PDB, where N is derived by the number of BINDING residues (they need to match the number of CRYPTIC-BINDING)
2. non-binding: if protein contains M binding/cryptic-binding residues, randomly sample M non-binding

In [2]:
import random

def load_row(row):
    protein_id = row[0] + row[1]
    annotation = row[3].split(' ')
    annotation = [int(i[1:]) for i in annotation]
    sequence = row[4]
    
    # load embeddings for this protein from all embedding spaces
    these_embeddings = {}
    length, have_same_length = 0, True
    for i, (embeddings_name, embeddings_path) in enumerate(EMB_SPACES):
        path = f"{embeddings_path}/{protein_id}.npy"
        if not os.path.exists(path):
            have_same_length = False
            break
        embedding = np.load(path)
        these_embeddings[embeddings_name] = embedding
        # check that all embedding spaces have the same length for this protein
        if i == 0:
            length = embedding.shape[0]
        else:
            if embedding.shape[0] != length:
                have_same_length = False
                break
    
    # check that all embedding spaces have the same length for this protein
    if not have_same_length:
        return None, None, None, None

    return protein_id, annotation, sequence, these_embeddings

def shuffle_residues(length):
    randomly_ordered_residues = list(range(length))
    random.shuffle(randomly_ordered_residues)
    return randomly_ordered_residues

def get_embeddings(embeddings, these_embeddings, embedding_indices):
    for embeddings_name in these_embeddings:
        if embeddings_name not in embeddings:
            embeddings[embeddings_name] = []
        
        collected_embeddings = these_embeddings[embeddings_name][embedding_indices]
        embeddings[embeddings_name].append(collected_embeddings)

def add_protein_id(protein_id, protein_ids, type):
    assert protein_id not in protein_ids, f"Protein {protein_id} is already added"
    protein_ids.add(f'{protein_id}_{type}')

### Collect cryptobench and balanced scPDB

In [24]:
# collect data from the cryptobench dataset to count the number of binding residues in it
protein_ids = set()
feature_data = []
embeddings = {}
from  constants import CRYPTOBENCH_TEST_DATASET, LIGYSIS_DATASET
number_of_cryptic_residues = 0
number_of_non_binding_residues = 0

# with open(CRYPTOBENCH_TEST_DATASET, 'r') as f:
with open(CRYPTOBENCH_TRAIN_DATASET, 'r') as f:
    reader = csv.reader(f, delimiter=';')
    for row in reader:
        protein_id, annotation, sequence, these_embeddings = load_row(row)
        if protein_id is None:
            continue
        for i in annotation:
            feature_data.append([sequence[i], 'CRYPTIC-BINDING'])
            number_of_cryptic_residues += 1
        # add NON-BINDING residues until we have the same number of NON-BINDING and BINDING residues in the embeddings (loop randomly the non-binding residues)
        randomly_ordered_residues = shuffle_residues(len(sequence))
        embedding_indices = annotation.copy()  # start with the binding residues
        for i, residue_index in enumerate(randomly_ordered_residues):
            if residue_index in annotation: # skip binding residues
                continue
            if i > round(len(annotation) / 2):
                break
            feature_data.append([sequence[residue_index], 'NON-BINDING'])
            embedding_indices.append(i)
            number_of_non_binding_residues += 1
        
        add_protein_id(protein_id, protein_ids, 'CRYPTIC')

        get_embeddings(embeddings, these_embeddings, embedding_indices)

import pickle
# this is a list of scPDB protein ids that have seq similarity > 10 % with the LIGYSIS dataset
# with open('/home/unix/vkrhk/EmmaEmb/data/banned_protein_ids.pkl', 'rb') as f:
#     banned_protein_ids = pickle.load(f)

number_of_regular_binding_residues = 0

# with open(LIGYSIS_DATASET, 'r') as f:
with open(SCPDB_DATASET, 'r') as f:
    reader = csv.reader(f, delimiter=';')
    for row in reader:
        if number_of_regular_binding_residues >= number_of_cryptic_residues:
            break

        protein_id = row[0] + row[1]
        protein_id, annotation, sequence, these_embeddings = load_row(row)
        if protein_id is None:
            continue
        # if protein_id in banned_protein_ids:
        #     continue

        for i in annotation:
            feature_data.append([sequence[i], 'BINDING'])
            number_of_regular_binding_residues += 1

        randomly_ordered_residues = shuffle_residues(len(sequence))
        embedding_indices = annotation.copy()  # start with the binding residues
        this_number_of_non_binding_residues = 0
        for residue_index in randomly_ordered_residues:
            
            if residue_index in annotation: # skip binding residues
                continue
            if this_number_of_non_binding_residues > round(len(annotation) / 2) or \
                number_of_cryptic_residues <= number_of_non_binding_residues:
                break
            
            feature_data.append([sequence[residue_index], 'NON-BINDING'])
            embedding_indices.append(residue_index)
            number_of_non_binding_residues += 1
            this_number_of_non_binding_residues += 1

        add_protein_id(protein_id, protein_ids, 'REGULAR')
        get_embeddings(embeddings, these_embeddings, embedding_indices)

for embeddings_name in embeddings:
    concatenated_embeddings_path = f"{EMBEDDINGS_PATH}/concatenated-embeddings/{embeddings_name}_binding_site_embeddings.npy"
    embeddings[embeddings_name] = np.concatenate(embeddings[embeddings_name], axis=0)
    np.save(concatenated_embeddings_path, embeddings[embeddings_name])  

feature_data = pd.DataFrame.from_records(feature_data, columns=["amino acid", "binding_site"])

# initiate Emma object and load embedding spaces
emma = Emma(feature_data=feature_data)
for embeddings_name, _ in EMB_SPACES:
    emma.add_emb_space(
        embeddings_source=f"{EMBEDDINGS_PATH}/concatenated-embeddings/{embeddings_name}_binding_site_embeddings.npy",
        emb_space_name=embeddings_name)

38921 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.
Embedding space 'ANKH' added successfully.
Embeddings have 1536 features each.
Embedding space 'ProstT5' added successfully.
Embeddings have 1024 features each.
Embedding space 'ProtT5' added successfully.
Embeddings have 1024 features each.
Embedding space 'ESM1' added successfully.
Embeddings have 1280 features each.
Embedding space 'ESMC' added successfully.
Embeddings have 960 features each.


In [20]:
SCPDB_DATASET

'/home/unix/vkrhk/EmmaEmb/data/scPDB_enhanced_binding_sites_translated.csv'

In [19]:
with open(f'{DATA_PATH}/protein_ids.pkl', 'rb') as f:
    print(pickle.load(f))

{'6oahF_CRYPTIC', '5ellB_CRYPTIC', '3d76A_REGULAR', '4ud4B_CRYPTIC', '3p08B_CRYPTIC', '3bv9B_CRYPTIC', '1j8fC_CRYPTIC', '7f0oA_CRYPTIC', '3qasA_CRYPTIC', '8g25H_CRYPTIC', '5h61B_CRYPTIC', '3vdmB_CRYPTIC', '1tqdA_CRYPTIC', '3qrsA_REGULAR', '5un3A_CRYPTIC', '4lrzA_REGULAR', '1efhB_CRYPTIC', '7wyoB_CRYPTIC', '6s3pA_CRYPTIC', '5f7qC_CRYPTIC', '8bp9BBB_CRYPTIC', '4ekfA_CRYPTIC', '4gf1B_CRYPTIC', '4choB_REGULAR', '7jrbA_CRYPTIC', '7djnA_CRYPTIC', '5eo7A_CRYPTIC', '2qa2A_REGULAR', '3c06A_REGULAR', '3s1cA_REGULAR', '3rzpC_REGULAR', '4d2oB_CRYPTIC', '6xl4C_CRYPTIC', '1eu1A_REGULAR', '3pgsB_CRYPTIC', '4kfgB_REGULAR', '2xh9B_REGULAR', '1txzA_REGULAR', '1jhoA_REGULAR', '1kc1A_REGULAR', '2hdqA_REGULAR', '6jnjA_CRYPTIC', '1gtrA_REGULAR', '3rv9A_REGULAR', '4tqrA_CRYPTIC', '4v0sA_REGULAR', '3a7gA_CRYPTIC', '5wm0A_CRYPTIC', '5upjA_REGULAR', '1fl1B_CRYPTIC', '4d1nA_CRYPTIC', '4nfcA_CRYPTIC', '4glxA_REGULAR', '2h3wA_REGULAR', '8i7xC_CRYPTIC', '5d27A_CRYPTIC', '9bhrB_CRYPTIC', '5k3aA_CRYPTIC', '5z67A_CRYP

In [26]:
import pickle
# with open(f'{DATA_PATH}/test_protein_ids.pkl', 'wb') as f:
with open(f'{DATA_PATH}/protein_ids.pkl', 'wb') as f:
    pickle.dump(protein_ids, f)

In [25]:
# SANITY CHECK
print(f'CRYPTIC residues:     {feature_data[feature_data["binding_site"] == "CRYPTIC-BINDING"].shape[0]}')
print(f'BINDING residues:     {feature_data[feature_data["binding_site"] == "BINDING"].shape[0]}')
print(f'NON-BINDING residues: {feature_data[feature_data["binding_site"] == "NON-BINDING"].shape[0]}')
print()
print(f'Number of proteins:   {len(protein_ids)}')

CRYPTIC residues:     12964
BINDING residues:     12970
NON-BINDING residues: 12964

Number of proteins:   1082


In [36]:
from emmaemb.vizualisation import plot_knn_alignment_across_classes

for embeddings_name, _ in EMB_SPACES:
    emma.calculate_pairwise_distances(emb_space=embeddings_name, metric='euclidean')

fig_3 = plot_knn_alignment_across_classes(
    emma, k=3, metric='euclidean', feature='binding_site')

fig_3.update_layout(coloraxis_cmin=0, coloraxis_cmax=1)
fig_3.show()

Calculating pairwise distances using euclidean...
Using GPU for distance calculation.


Computing pairwise distances (euclidean): 100%|████████████████████████████████| 389/389 [00:03<00:00, 107.49it/s]


Calculating pairwise distances using euclidean...
Using GPU for distance calculation.


Computing pairwise distances (euclidean): 100%|████████████████████████████████| 389/389 [00:03<00:00, 106.19it/s]


Calculating pairwise distances using euclidean...
Using GPU for distance calculation.


Computing pairwise distances (euclidean): 100%|████████████████████████████████| 389/389 [00:01<00:00, 345.23it/s]


Calculating pairwise distances using euclidean...
Using GPU for distance calculation.


Computing pairwise distances (euclidean): 100%|████████████████████████████████| 389/389 [00:01<00:00, 346.90it/s]


### Reshuffle
Take the undersampled dataset and randomly reshuffle the labels

In [45]:
# Randomly reshuffle binding_site labels in emma metadata
emma.metadata["binding_site"] = np.random.permutation(emma.metadata["binding_site"].to_numpy())

for embeddings_name, _ in emb_spaces:
    emma.calculate_pairwise_distances(emb_space=embeddings_name, metric='euclidean')

fig_3 = plot_knn_alignment_across_classes(
    emma, k=3, metric='euclidean', feature='binding_site')

fig_3.update_layout(coloraxis_cmin=0, coloraxis_cmax=1)
fig_3.show()

Calculating pairwise distances using euclidean...
Using GPU for distance calculation.


Computing pairwise distances (euclidean):   0%|                                                                                              | 0/390 [00:00<?, ?it/s]

Computing pairwise distances (euclidean): 100%|███████████████████████████████████████████████████████████████████████████████████| 390/390 [00:03<00:00, 120.62it/s]


Calculating pairwise distances using euclidean...
Using GPU for distance calculation.


Computing pairwise distances (euclidean): 100%|███████████████████████████████████████████████████████████████████████████████████| 390/390 [00:02<00:00, 135.08it/s]


...

Computing pairwise distances (euclidean): 100%|███████████████████████████████████████████████████████████████████████████████████| 390/390 [00:01<00:00, 358.78it/s]


Calculating pairwise distances using euclidean...
Using GPU for distance calculation.


Computing pairwise distances (euclidean): 100%|███████████████████████████████████████████████████████████████████████████████████| 390/390 [00:01<00:00, 311.53it/s]


Calculating pairwise distances using euclidean...
Using GPU for distance calculation.


Computing pairwise distances (euclidean): 100%|███████████████████████████████████████████████████████████████████████████████████| 390/390 [00:01<00:00, 373.25it/s]


### Whole dataset
Take the kNN scores computed externally for the whole dataset (balanced regular/cryptic, but including ALL non-binding) and plot the heatmap

In [5]:
import pandas as pd
import plotly.express as px

def plot_scores(path):
    k=3
    metric = 'euclidean'
    feature = 'binding_site'

    performances = []
    for file in os.listdir(path):
        if not file.endswith('.csv'):
            continue
        performance = pd.read_csv(f'{path}/{file}', delimiter=',')
        emb_space = performance.columns[1]
        regular_binding_score = performance.iloc[0][emb_space]
        cryptic_binding_score = performance.iloc[1][emb_space]
        non_binding_score = performance.iloc[2][emb_space]
        performances.append((emb_space, regular_binding_score, cryptic_binding_score, non_binding_score))

    performances = pd.DataFrame(performances, columns=['Embedding Space', \
                                                       performance.iloc[0][performance.columns[0]], \
                                                       performance.iloc[1][performance.columns[0]], \
                                                       performance.iloc[2][performance.columns[0]]])
    heatmap_data = performances.set_index('Embedding Space').T

    fig = px.imshow(
        heatmap_data,
        labels={
            "x": "Embedding Space",
            "y": "Feature Class",
            "color": "Mean KNN feature alignment score",
        },
        title=f"Mean KNN feature alignment scores for {feature}<br>k = {k}, {metric}, imbalanced",
        color_continuous_scale=[
        (0.0, "lightblue"),
        (1.0, '#303496'),],
        text_auto=".2f",
        aspect="auto",
        range_color=[0,1],  
    )

    fig.update_layout(font=dict(family="Arial"))
    fig.show()
plot_scores(f'{IMG_OUTPUT_PATH}/knn-binding-sites')

### Randomly shuffled whole dataset
Same as above, just randomly reshuffle the labels (externally calculated - `analysis/knn.py`).

From both shuffled datasets is apparent that when no underlying biological structure is present (randomly assigned labels), the kNN alignment score reflects the class weight distribution. 

In [ ]:
plot_scores(f'{IMG_OUTPUT_PATH}/knn-binding-sites-RANDOM')   

# Number of cryptic residues: 12964
# Number of regular binding residues: 12993
# Number of non-binding residues: 286242
# Percentage of cryptic residues: 0.04152479668416619
# Percentage of regular binding residues: 0.04161768615530479
# Percentage of non-binding residues: 0.916857517160529

In [1]:
import csv
import os
import sys
import pickle 
import pandas as pd
import numpy as np
import argparse
sys.path.append('/home/unix/vkrhk/EmmaEmb/EmmaEmb/analysis')
from dim_reduction_utils import load_balanced_cryptic_and_regular_data, load_dataset_with_all_balanced_classes
from emmaemb.core import Emma
from emmaemb.vizualisation import plot_knn_class_mixing_matrix
from constants import DATA_PATH, EMBEDDINGS_PATH, IMG_OUTPUT_PATH, EMB_SPACES, CRYPTOBENCH_TRAIN_DATASET, SCPDB_DATASET

emma = load_dataset_with_all_balanced_classes()

metric = 'euclidean'
n_trees = 100
embedding_space = 'ESM2'
# emma.build_annoy_index(emb_space=embedding_space, metric=metric, n_trees=n_trees)


38898 samples loaded.
Categories in meta data: ['binding_site']
Numerical columns in meta data: []
Embedding space 'ESM2' added successfully.
Embeddings have 1280 features each.
Embedding space 'ANKH' added successfully.
Embeddings have 1536 features each.
Embedding space 'ProstT5' added successfully.
Embeddings have 1024 features each.
Embedding space 'ProtT5' added successfully.
Embeddings have 1024 features each.


In [4]:
from emmaemb.vizualisation import plot_knn_class_mixing_matrix
metric = 'euclidean'
n_trees = 100
embedding_space = 'ESM2'
# emma.build_annoy_index(emb_space=embedding_space, metric=metric, n_trees=n_trees)
fig_class_mixing_matrix = plot_knn_class_mixing_matrix(
    emma,
    emb_space=embedding_space,
    feature="binding_site",
    k=3,
    metric=metric,
    use_annoy=True,
    annoy_metric=metric,
    n_trees=n_trees
)
fig_class_mixing_matrix.update_layout(height=600, width=600)
fig_class_mixing_matrix.show()

Using Annoy index with 100 trees and euclidean metric.
